# Autoencoder Analysis

Evaluate a pretrained convolutional autoencoder on STL-10.
Sections: training trajectories, evaluation metrics (k-NN + linear probe),
intermediate feature maps, and reconstruction visualization.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from models.autoencoder import Autoencoder
from evaluation import extract_features
from evaluation.knn import knn_classify
from evaluation.linear_probe import LinearProbe
from utils.data import get_eval_loaders

sns.set_theme(style="whitegrid", font_scale=1.1)
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]
RESULTS_DIR = PROJECT_ROOT / "results" / "autoencoder"
print(f"Device: {device}")

## Section A: Training Trajectories

In [ ]:
with open(RESULTS_DIR / "history.json") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]
knn_accs = [h["knn_top1"] for h in history]
lrs = [h["lr"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, losses, color="steelblue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")

axes[1].plot(epochs, [a * 100 for a in knn_accs], color="darkorange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("k-NN Accuracy (%)")
axes[1].set_title("k-NN Top-1 Accuracy")

axes[2].plot(epochs, lrs, color="seagreen")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule")

fig.suptitle("Autoencoder Training Trajectories", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Best k-NN accuracy: {max(knn_accs)*100:.2f}% (epoch {epochs[np.argmax(knn_accs)]})")
print(f"Final loss: {losses[-1]:.6f}")

## Section B: Evaluation Results

In [ ]:
# Load trained model
ckpt = torch.load(RESULTS_DIR / "best.pt", map_location=device, weights_only=True)
model = Autoencoder(backbone="resnet18")
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Loaded checkpoint from {RESULTS_DIR / 'best.pt'}")
print(f"Latent dim: {model.latent_dim}")

In [ ]:
# Autoencoder uses unnormalized [0,1] images (decoder has Sigmoid output)
ae_transform = transforms.Compose([
    transforms.Resize(96),
    transforms.CenterCrop(96),
    transforms.ToTensor(),
])
loaders = get_eval_loaders(batch_size=256, transform=ae_transform)

print(f"Train set size: {len(loaders['train'].dataset)}")
print(f"Test set size:  {len(loaders['test'].dataset)}")
print(f"Low-data size:  {len(loaders['train_lowdata'].dataset)}")

In [ ]:
# Extract features once for k-NN sweep
train_features, train_labels = extract_features(model.encoder, loaders["train"], device)
test_features, test_labels = extract_features(model.encoder, loaders["test"], device)

# L2-normalize
train_features_norm = F.normalize(train_features, dim=1)
test_features_norm = F.normalize(test_features, dim=1)

print(f"Train features: {tuple(train_features.shape)}")
print(f"Test features:  {tuple(test_features.shape)}")

In [ ]:
# k-NN sweep
k_values = [1, 5, 10, 20, 50, 100, 200]
knn_results = {}

for k in k_values:
    acc = knn_classify(train_features_norm, train_labels, test_features_norm, test_labels, k=k)
    knn_results[k] = acc
    print(f"k={k:>3d}  accuracy: {acc*100:.2f}%")

# Plot k-NN accuracy vs k
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(knn_results.keys()), [v * 100 for v in knn_results.values()],
        "o-", color="steelblue", label="Autoencoder")
ax.set_xscale("log")
ax.set_xlabel("k")
ax.set_ylabel("Accuracy (%)")
ax.set_title("k-NN Accuracy vs k (Autoencoder)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Linear probe -- full data
probe_full = LinearProbe(feature_dim=model.latent_dim, num_classes=10, device=device)
probe_full_losses = probe_full.fit(model.encoder, loaders["train"])
probe_full_acc = probe_full.evaluate(model.encoder, loaders["test"])
print(f"Linear probe (full data) accuracy: {probe_full_acc*100:.2f}%")

In [ ]:
# Linear probe -- low data (1%)
probe_low = LinearProbe(feature_dim=model.latent_dim, num_classes=10, device=device)
probe_low_losses = probe_low.fit(model.encoder, loaders["train_lowdata"])
probe_low_acc = probe_low.evaluate(model.encoder, loaders["test"])
print(f"Linear probe (low data) accuracy:  {probe_low_acc*100:.2f}%")

In [ ]:
# Summary table
best_knn_k = max(knn_results, key=knn_results.get)
best_knn_acc = knn_results[best_knn_k]

print(f"{'Metric':<30s} {'Value':>12s}")
print("-" * 44)
print(f"{'k-NN (best, k=' + str(best_knn_k) + ')':<30s} {best_knn_acc*100:>11.2f}%")
print(f"{'k-NN (k=20)':<30s} {knn_results[20]*100:>11.2f}%")
print(f"{'Linear probe (full data)':<30s} {probe_full_acc*100:>11.2f}%")
print(f"{'Linear probe (1% data)':<30s} {probe_low_acc*100:>11.2f}%")

## Section C: Intermediate Layer Feature Maps

In [ ]:
# Get a single test image
test_batch, test_labels_batch = next(iter(loaders["test"]))
sample_img = test_batch[0:1].to(device)
sample_label = test_labels_batch[0].item()

# Register hooks on ResNet-18 intermediate layers
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = []
for name in ["layer1", "layer2", "layer3", "layer4"]:
    h = getattr(model.encoder, name).register_forward_hook(make_hook(name))
    hooks.append(h)

# Forward pass to collect activations
with torch.no_grad():
    _ = model.encoder(sample_img)

# Remove hooks
for h in hooks:
    h.remove()

print(f"Sample image class: {CLASS_NAMES[sample_label]}")
for name, act in activations.items():
    print(f"{name}: {tuple(act.shape)}")

In [ ]:
# Visualize first 16 channels for each layer
layer_names = ["layer1", "layer2", "layer3", "layer4"]
layer_channels = [64, 128, 256, 512]

orig_img = test_batch[0].permute(1, 2, 0).numpy().clip(0, 1)

for layer_name, n_ch in zip(layer_names, layer_channels):
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    fig.suptitle(f"{layer_name} ({n_ch} channels) -- first 16 shown", fontsize=14)

    # Show original image in top-left
    axes[0, 0].imshow(orig_img)
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    act = activations[layer_name][0]  # (C, H, W)
    idx = 0
    for i in range(4):
        for j in range(5):
            if i == 0 and j == 0:
                continue
            ax = axes[i, j]
            if idx < 16:
                ax.imshow(act[idx].numpy(), cmap="viridis")
                ax.set_title(f"Ch {idx}")
                idx += 1
            ax.axis("off")

    plt.tight_layout()
    plt.show()

## Section D: Reconstruction Visualization

In [ ]:
# Get 8 test images and reconstruct
batch = test_batch[:8].to(device)
labels = test_labels_batch[:8]

with torch.no_grad():
    x_hat, z = model(batch)

batch_cpu = batch.cpu()
x_hat_cpu = x_hat.cpu()

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Autoencoder Reconstructions (top: original, bottom: reconstruction)", fontsize=14)

for i in range(8):
    # Original
    img_orig = batch_cpu[i].permute(1, 2, 0).numpy().clip(0, 1)
    axes[0, i].imshow(img_orig)
    axes[0, i].set_title(CLASS_NAMES[labels[i].item()], fontsize=10)
    axes[0, i].axis("off")

    # Reconstruction
    img_recon = x_hat_cpu[i].permute(1, 2, 0).numpy().clip(0, 1)
    axes[1, i].imshow(img_recon)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Reconstruction", fontsize=12)

plt.tight_layout()
plt.show()

# Compute reconstruction MSE
mse = F.mse_loss(x_hat_cpu, batch_cpu).item()
print(f"Reconstruction MSE (8 samples): {mse:.6f}")